# Modelagem e Avaliação — Rain in Australia

**PCO213 — Aprendizado de Máquina e Mineração de Dados | UNIFEI**

Este notebook cobre as **Etapas 3, 4 e 5** do pipeline:
1. Pré-processamento e split treino/teste
2. Validação cruzada dos modelos
3. Tuning de hiperparâmetros
4. Avaliação final no conjunto de teste
5. Geração de figuras 7–10 e exportação do melhor modelo

**Regra de data leakage:** todas as transformações (imputação, encoding, scaling)
são ajustadas exclusivamente no conjunto de treino, dentro de `Pipeline`/`ColumnTransformer`.

In [ ]:
import sys
from pathlib import Path

def _find_project_root() -> Path:
    for candidate in [Path('.'), Path('..')]:
        resolved = candidate.resolve()
        if (resolved / 'src').exists() and (resolved / 'requirements.txt').exists():
            return resolved
    raise FileNotFoundError("Raiz do projeto nao encontrada.")

_root = _find_project_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

print(f"Raiz do projeto: {_root}")

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)

---
## Seção 1 — Carregamento e Pré-processamento

In [ ]:
from src.data_loader import load_raw
from src.preprocessing import clean_data, load_processed, split_data, get_feature_lists

# Carrega dataset bruto
df_raw = load_raw()

# Limpeza estrutural (ou carrega cached se já existir)
df_clean = load_processed(df_raw)
print(f"\nShape limpo: {df_clean.shape}")

In [ ]:
# Split treino/teste estratificado (80/20, random_state=42)
X_train, X_test, y_train, y_test = split_data(df_clean)

# Identifica features APENAS a partir do treino (sem leakage)
numeric_features, categorical_features = get_feature_lists(X_train)

---
## Seção 2 — Definição dos Modelos

Cada modelo é encapsulado em um `Pipeline` completo:
`ColumnTransformer (preprocessor)` → `Classifier`.
Nenhum transformador é ajustado aqui — apenas criado.

In [ ]:
from src.modeling import get_models, get_svm_sample

models = get_models(numeric_features, categorical_features)
print("\nModelos prontos para treinamento:")
for name, pipe in models.items():
    clf = pipe.named_steps['classifier']
    print(f"  {name:<15}: {type(clf).__name__}")

In [ ]:
# Amostra estratificada do treino para o LinearSVC
# (SVC RBF inviavel em ~142k linhas; LinearSVC em amostra e eficiente)
X_svm, y_svm = get_svm_sample(X_train, y_train)

---
## Seção 3 — Validação Cruzada (StratifiedKFold, k=5)

Avalia cada modelo com 5 folds estratificados.
Métricas: F1, ROC-AUC, Precision, Recall (média ± desvio padrão).

In [ ]:
from src.modeling import cross_validate_models

df_cv = cross_validate_models(models, X_train, y_train, X_svm, y_svm)
df_cv

---
## Seção 4 — Tuning de Hiperparâmetros

- **DecisionTree**: `GridSearchCV` (max_depth, min_samples_split, min_samples_leaf, criterion)
- **RandomForest**: `RandomizedSearchCV` (n_estimators, max_depth, min_samples_*)
- **LogisticRegression**: `GridSearchCV` simples (C)
- **Baseline / SVM**: sem tuning

> Atenção: esta célula pode levar vários minutos dependendo do hardware.

In [ ]:
from src.modeling import tune_hyperparameters

tuned_models = tune_hyperparameters(models, X_train, y_train, X_svm, y_svm)

---
## Seção 5 — Avaliação Final no Conjunto de Teste

O conjunto de teste é usado **apenas aqui** — nunca antes.

In [ ]:
from src.evaluation import evaluate_on_test, build_metrics_table, print_classification_reports

results = []
for name, pipeline in tuned_models.items():
    print(f"Avaliando {name}...")
    res = evaluate_on_test(name, pipeline, X_test, y_test)
    results.append(res)

print("\n" + "="*60)
print("TABELA COMPARATIVA DE METRICAS")
print("="*60)
df_metrics = build_metrics_table(results)

In [ ]:
print_classification_reports(results, y_test)

---
## Seção 6 — Seleção e Exportação do Melhor Modelo

In [ ]:
from src.evaluation import select_best_model, save_best_model

best_name, best_pipeline = select_best_model(df_metrics, tuned_models)
save_best_model(best_name, best_pipeline)

---
## Seção 7 — Figuras de Avaliação (Fig 7–10)

Todas salvas em `outputs/figures/` para uso no artigo e nos slides.

In [ ]:
from src.evaluation import plot_confusion_matrix

best_result = next(r for r in results if r['modelo'] == best_name)
fig7 = plot_confusion_matrix(best_name, y_test, best_result['y_pred'])
# Salvo em outputs/figures/07_confusion_matrix.png

In [ ]:
from src.evaluation import plot_roc_curves

fig8 = plot_roc_curves(results, y_test)
# Salvo em outputs/figures/08_roc_curves.png

In [ ]:
from src.evaluation import plot_pr_curve

if best_result['y_prob'] is not None:
    fig_pr = plot_pr_curve(best_name, y_test, best_result['y_prob'])
    # Salvo em outputs/figures/09_pr_curve.png
else:
    print("Curva PR ignorada: modelo nao fornece probabilidades.")

In [ ]:
from src.evaluation import plot_metrics_comparison

fig9 = plot_metrics_comparison(df_metrics)
# Salvo em outputs/figures/09_metrics_comparison.png

In [ ]:
from src.evaluation import feature_importance

fig10 = feature_importance(results, tuned_models, X_train)
# Salvo em outputs/figures/10_feature_importance.png

In [ ]:
from src.config import FIGURES_DIR, TABLES_DIR, MODELS_DIR

print("="*55)
print("SUMARIO FINAL")
print("="*55)
print(f"\nMelhor modelo: {best_name}")
print(f"\nFiguras geradas ({FIGURES_DIR}):")
for f in sorted(FIGURES_DIR.glob('*.png')):
    print(f"  {f.name}")
print(f"\nTabelas ({TABLES_DIR}):")
for f in sorted(TABLES_DIR.glob('*.csv')):
    print(f"  {f.name}")
print(f"\nModelo salvo ({MODELS_DIR}):")
for f in sorted(MODELS_DIR.glob('*.joblib')):
    print(f"  {f.name}")